In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Optional

import pandas as pd
import mne

# ============================================================
# CONFIG
# ============================================================
BIDS_ROOT = Path(r"D:\HUP dataset")
MIN_SFREQ = 512.0
OUTPUT_CSV = Path(r"D:\HUP dataset\hup_interictal_sampling_audit.csv")
OUTPUT_FILTERED_CSV = Path(r"D:\HUP dataset\hup_interictal_sampling_gt512.csv")

# If you want to restrict to a subset, put subject IDs here, e.g.
# SUBJECT_FILTER = {"sub-HUP070", "sub-HUP097"}
SUBJECT_FILTER: Optional[set[str]] = None


# ============================================================
# HELPERS
# ============================================================
def parse_entity(text: str, key: str) -> Optional[str]:
    m = re.search(rf"_{key}-([^_]+)", text)
    return m.group(1) if m else None


def find_sidecar(edf_path: Path, suffix: str) -> Optional[Path]:
    p = edf_path.with_name(edf_path.name.replace("_ieeg.edf", suffix))
    return p if p.exists() else None


def read_json(path: Optional[Path]) -> dict:
    if path is None or not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_tsv(path: Optional[Path]) -> pd.DataFrame:
    if path is None or not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t")


def count_good_bad_channels(ch_df: pd.DataFrame) -> tuple[int, int]:
    if ch_df.empty or "status" not in ch_df.columns:
        return 0, 0
    status = ch_df["status"].astype(str).str.lower()
    good = int((status == "good").sum())
    bad = int((status == "bad").sum())
    return good, bad


# ============================================================
# MAIN AUDIT
# ============================================================
def main() -> None:
    rows = []

    for edf_path in sorted(BIDS_ROOT.rglob("*_ieeg.edf")):
        subject = re.search(r"(sub-[A-Za-z0-9]+)", str(edf_path))
        subject = subject.group(1) if subject else None
        if SUBJECT_FILTER is not None and subject not in SUBJECT_FILTER:
            continue

        task = parse_entity(edf_path.name, "task")
        if task is None or "interictal" not in task.lower():
            continue

        session = parse_entity(edf_path.name, "ses")
        acquisition = parse_entity(edf_path.name, "acq")
        run = parse_entity(edf_path.name, "run")

        json_path = find_sidecar(edf_path, "_ieeg.json")
        channels_path = find_sidecar(edf_path, "_channels.tsv")
        events_path = find_sidecar(edf_path, "_events.tsv")

        meta = read_json(json_path)
        ch_df = read_tsv(channels_path)
        good_ch, bad_ch = count_good_bad_channels(ch_df)

        # Try sidecar first, then raw header
        sfreq = meta.get("SamplingFrequency")
        if sfreq is None:
            try:
                raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="ERROR")
                sfreq = float(raw.info["sfreq"])
                n_channels = len(raw.ch_names)
                duration_sec = float(raw.n_times / raw.info["sfreq"])
            except Exception as e:
                sfreq = None
                n_channels = None
                duration_sec = None
                read_error = str(e)
            else:
                read_error = ""
        else:
            try:
                raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="ERROR")
                n_channels = len(raw.ch_names)
                duration_sec = float(raw.n_times / raw.info["sfreq"])
                read_error = ""
            except Exception as e:
                n_channels = None
                duration_sec = None
                read_error = str(e)

        high_cutoff = meta.get("HighCutoff")
        low_cutoff = meta.get("LowCutoff")
        powerline = meta.get("PowerLineFrequency")

        rows.append(
            {
                "subject": subject,
                "session": session,
                "task": task,
                "acquisition": acquisition,
                "run": run,
                "edf_path": str(edf_path),
                "json_exists": json_path is not None,
                "channels_exists": channels_path is not None,
                "events_exists": events_path is not None,
                "sampling_frequency_hz": float(sfreq) if sfreq is not None else None,
                "high_cutoff": high_cutoff,
                "low_cutoff": low_cutoff,
                "powerline_frequency": powerline,
                "n_channels_raw": n_channels,
                "n_good_channels": good_ch,
                "n_bad_channels": bad_ch,
                "duration_sec": duration_sec,
                "passes_sfreq_gt_512": bool(sfreq is not None and float(sfreq) > MIN_SFREQ),
                "read_error": read_error,
            }
        )

    if not rows:
        print("No interictal EDF runs found.")
        return

    df = pd.DataFrame(rows)
    df = df.sort_values(
        by=["passes_sfreq_gt_512", "sampling_frequency_hz", "subject", "run"],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)

    passed = df[df["passes_sfreq_gt_512"]].copy()

    # subject-level summary
    subj_summary = (
        df.groupby("subject", dropna=False)
        .agg(
            n_interictal_runs=("edf_path", "count"),
            n_runs_gt_512=("passes_sfreq_gt_512", "sum"),
            max_sfreq_hz=("sampling_frequency_hz", "max"),
            min_sfreq_hz=("sampling_frequency_hz", "min"),
        )
        .reset_index()
        .sort_values(by=["n_runs_gt_512", "max_sfreq_hz", "subject"], ascending=[False, False, True])
    )

    print("\n=== SUBJECT SUMMARY ===")
    print(subj_summary.to_string(index=False))

    print("\n=== RUNS WITH SAMPLING FREQUENCY > 512 HZ ===")
    if passed.empty:
        print("None found.")
    else:
        print(
            passed[
                [
                    "subject",
                    "task",
                    "acquisition",
                    "run",
                    "sampling_frequency_hz",
                    "high_cutoff",
                    "n_good_channels",
                    "n_bad_channels",
                    "events_exists",
                ]
            ].to_string(index=False)
        )

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False)
    passed.to_csv(OUTPUT_FILTERED_CSV, index=False)

    subj_summary_path = OUTPUT_CSV.with_name("hup_interictal_subject_summary.csv")
    subj_summary.to_csv(subj_summary_path, index=False)

    print(f"\nSaved full audit to: {OUTPUT_CSV}")
    print(f"Saved filtered runs to: {OUTPUT_FILTERED_CSV}")
    print(f"Saved subject summary to: {subj_summary_path}")


if __name__ == "__main__":
    main()



=== SUBJECT SUMMARY ===
   subject  n_interictal_runs  n_runs_gt_512  max_sfreq_hz  min_sfreq_hz
sub-HUP126                  2              2        1024.0        1024.0
sub-HUP130                  2              2        1024.0        1024.0
sub-HUP134                  2              2        1024.0        1024.0
sub-HUP138                  2              2        1024.0        1024.0
sub-HUP139                  2              2        1024.0        1024.0
sub-HUP140                  2              2        1024.0        1024.0
sub-HUP146                  2              2        1024.0        1024.0
sub-HUP157                  2              2        1024.0        1024.0
sub-HUP160                  2              2        1024.0        1024.0
sub-HUP162                  2              2        1024.0        1024.0
sub-HUP163                  2              2        1024.0        1024.0
sub-HUP164                  2              2        1024.0        1024.0
sub-HUP165                

In [4]:
import numpy as np

with np.load('D:\HUP_processed\pipeline_b_car_0p5_120_128\sub-HUP070\sub-HUP070_ses-presurgery_task-ictal_acq-ecog_run-01.npz', allow_pickle=True) as data:
    print("Keys:", data.files)
    for key in data.files:
        print(f"\n{key}:")
        print(data[key])
        print("shape:", getattr(data[key], "shape", None))
        print("dtype:", getattr(data[key], "dtype", None))

Keys: ['X', 'y', 'y_train', 't_bounds', 'mask', 'rms_z', 'peak_z']

X:
[[[ 6.87213317e-02 -4.56562974e-02 -2.04069644e-01 ...  1.77478209e-01
    6.96329832e-01  1.06560516e+00]
  [ 3.35748553e-01  3.58032510e-02 -8.33366737e-02 ... -1.51896465e+00
   -5.56275368e-01  6.86385632e-01]
  [-1.98732466e-01  2.77819812e-01 -8.87119770e-02 ...  2.62340069e-01
    1.77901134e-01 -1.11898623e-01]
  ...
  [ 1.46964923e-01  5.84809422e-01  1.10348202e-01 ... -1.94470108e-01
    1.18891013e+00  2.11079764e+00]
  [ 5.44752032e-02 -1.30348384e-01  2.70116627e-01 ...  3.88618022e-01
    7.68711746e-01  1.17346394e+00]
  [-3.66409421e-01 -3.95989627e-01  2.94496089e-01 ...  5.09142540e-02
   -4.61282097e-02 -1.87330663e-01]]

 [[-4.83863465e-02 -1.83091804e-01 -5.30037344e-01 ... -5.44865012e-01
   -3.96635123e-02  4.48297024e-01]
  [-8.42219114e-01 -9.00699437e-01 -5.56046903e-01 ... -1.58902800e+00
   -1.00988793e+00 -7.19589055e-01]
  [-3.05981249e-01 -5.91118276e-01 -6.15184724e-01 ... -8.2950681

In [5]:
with np.load('D:\HUP_processed\pipeline_b_car_0p5_120_128\sub-HUP070\sub-HUP070_ses-presurgery_task-ictal_acq-ecog_run-04.npz', allow_pickle=True) as data:
    print("Keys:", data.files)
    for key in data.files:
        print(f"\n{key}:")
        print(data[key])
        print("shape:", getattr(data[key], "shape", None))
        print("dtype:", getattr(data[key], "dtype", None))

Keys: ['X', 'y', 'y_train', 't_bounds', 'mask', 'rms_z', 'peak_z']

X:
[[[-3.71214479e-01 -8.35367262e-01 -1.13066113e+00 ... -8.33228350e-01
   -4.01629508e-01 -1.28877446e-01]
  [-2.59724818e-02 -2.56545424e-01  3.55354667e-01 ...  8.43292296e-01
    9.90802348e-01  4.43992376e-01]
  [-7.64572173e-02 -1.31932721e-01 -6.11793399e-01 ...  1.17269254e+00
    6.88793838e-01  1.45623088e-01]
  ...
  [ 1.90182835e-01  1.86161458e-01  2.01597154e-01 ...  4.37488317e-01
   -1.64225563e-01 -7.90522873e-01]
  [ 2.28692576e-01  3.37369084e-01  5.24365425e-01 ...  1.25807416e+00
    1.16758668e+00  9.39020336e-01]
  [ 4.69994769e-02  9.58827790e-03  1.81432143e-01 ...  1.76807547e+00
    1.15445852e+00  3.80294919e-01]]

 [[ 2.74642086e+00  2.47031021e+00  3.22133660e+00 ...  2.29490817e-01
   -3.09321493e-01 -2.81622022e-01]
  [ 1.47231531e+00  1.36620808e+00  1.50404656e+00 ...  1.72531158e-01
   -5.24436116e-01 -9.00497377e-01]
  [-2.07497969e-01 -3.55138063e-01 -4.92048949e-01 ... -7.4801927